Important Notes:

camis:
    - there are no nulls present, and all values were stripe of white spaces

dba:
    - There is one null value for a resturant that doesn't have any records

boro:
    - Sanity check that only resturants in the 5 boroughs exist.
    - check no outliers exist in the columns

In [1]:
%pip install pyarrow


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from transform import transform
pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_parquet("~/Desktop/Resturant_inspections/data/raw/inspections.parquet")

In [4]:
data = df.copy()

In [5]:
# restart the kernel first, so nothing stale is in memory


dim, insp, viol, quar = transform(data)

print("dim_restaurant:", dim.shape)
print("fct_inspection:", insp.shape)
print("fct_violation: ", viol.shape)
print("quarantine:    ", quar.shape)
print(quar["reject_reason"].value_counts())

dim_restaurant: (31264, 10)
fct_inspection: (96585, 10)
fct_violation:  (288068, 8)
quarantine:     (0, 36)
Series([], Name: count, dtype: int64)


In [6]:
print(data["camis"].isna().sum(), "null camis in raw")
print(insp["is_uninspected"].sum(), "uninspected restaurants")
print(insp["inspection_date"].isna().sum(), "null inspection dates")

0 null camis in raw
3712 uninspected restaurants
3712 null inspection dates


In [80]:
dict_cols = [c for c in data.columns
             if data[c].apply(lambda x: isinstance(x, (dict, list))).any()]

data = data.drop(columns = dict_cols)
data = data.drop_duplicates()

In [81]:
#camis
def clean_camis(data):
    data["camis"] = data["camis"].astype("string").str.strip().replace("", pd.NA)

    null_count = data["camis"].isna().sum()
    print(f"Null counts: {null_count}")
    only_numbers = data["camis"].str.fullmatch(r"\d+", na=False)
    print(f"Clean Data: {only_numbers.sum()} / {len(data)}")
    if not only_numbers.all():
        raise ValueError(f"{(~only_numbers).sum()} camis values are not digits only")

    print(f"Data type: {data['camis'].dtype}")

clean_camis(data)


Null counts: 0
Clean Data: 294513 / 294513
Data type: string


In [82]:
def clean_dba(data):
    data["dba"] = data["dba"].astype("string").str.strip().str.upper()
    data["dba"] = data["dba"].replace("", pd.NA)
    data["dba"] = data["dba"].fillna(data.groupby("camis")["dba"].transform("first"))

clean_dba(data)

In [83]:
def clean_boro(data):
    data["boro"] = data["boro"].astype("string").str.strip()
    valid_inputs = ["Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"]
    data["boro"] =  data["boro"].where(data["boro"].isin(valid_inputs), pd.NA)

clean_boro(data)
    

In [84]:
def clean_building_street(data):
   for col in ["building", "street"]:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.upper()
            .replace("", pd.NA)
        )
clean_building_street(data)
    

In [85]:
def clean_zipcode(data):
    data["zipcode"] = data["zipcode"].astype("string").str.strip()
    data["zipcode"] = data["zipcode"].where(data["zipcode"].str.fullmatch(r"\d{5}"), pd.NA)
    data["zipcode"] = data["zipcode"].fillna(data.groupby("camis")["zipcode"].transform("first"))

clean_zipcode(data)

In [86]:
def clean_lat_long(data):
    data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")
    data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")
    valid = (
        data["latitude"].between(40.4, 41.0)
        & data["longitude"].between(-74.3, -73.6)
    )

    data.loc[~valid, ["latitude", "longitude"]] = pd.NA

clean_lat_long(data)

In [87]:
def clean_community_info(data):
    for col in ["council_district", "community_board", "census_tract", "bin", "bbl", "nta"]:
        data[col] = data[col].astype("string").str.strip()

clean_community_info(data)

In [88]:
# came in as text, meaning we dont need to wory about the .0 if it was a float.

def clean_phone(data):
    data["phone"] = data["phone"].astype("string").str.strip().str.replace(r"\D", "", regex=True)
    data["phone"]= data["phone"].where(data["phone"].str.fullmatch(r"\d{10}", na=False))  # exactly 10 digits, else null
    data["phone"] = data["phone"].str.replace(
        r"(\d{3})(\d{3})(\d{4})", r"\1-\2-\3", regex=True   # format as xxx-xxx-xxxx
    )

clean_phone(data)   

In [89]:
def clean_cuisine(data):
    data["cuisine_description"].astype("string").str.strip().replace("", pd.NA)

clean_cuisine(data)

In [90]:
def clean_inspection_date(data):
    parsed = pd.to_datetime(data["inspection_date"], errors="coerce")
    today = pd.Timestamp.today().normalize()
    uninspected = parsed == pd.Timestamp("1900-01-01")
    invalid = parsed.isna()
    future = parsed > today
    data["is_uninspected"] = uninspected
    data["inspection_date"] = parsed.mask(uninspected)
    if "reject_reason" not in data.columns:
        data["reject_reason"] = pd.Series(pd.NA, index= data.index, dtype="string")
    data.loc[invalid & data["reject_reason"].isna(), "reject_reason"] = "invalid_inspection_date"
    data.loc[future & data["reject_reason"].isna(), "reject_reason"] = "future_inspection_date"

    
clean_inspection_date(data)

In [91]:
def clean_inspection_type(data):
    
    parts = (
        data["inspection_type"]
        .astype("string")
        .str.split(" / ", n=1, expand=True)
        .reindex(columns=[0, 1])
        .astype("string"))

    data["inspection_program"] = parts[0].str.strip().replace("", pd.NA).astype("category")
    data["inspection_type"] = parts[1].str.strip().replace("", pd.NA).astype("category")

clean_inspection_type(data)

In [92]:
def clean_action(data):
    data["action"] = (
        data["action"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

    data["is_closed"] = data["action"].str.contains(
        r"closed by dohmh", case=False, na=False
    )

clean_action(data)


In [93]:
def clean_score(data):
    data["score"] = pd.to_numeric(data["score"], errors="coerce")
    data["score"] = data["score"].mask(data["score"] < 0)

clean_score(data)

In [94]:
# must run after clean_score!!!!!!!def clean_grade_record_date(data):
def clean_grade_record_date(data):
    for col in ["grade_date", "record_date"]:
        raw = data[col]
        parsed = pd.to_datetime(raw, errors="coerce")

        failed = parsed.isna() & raw.notna()
        print(f"{col}: {failed.sum()} failed to parse, {raw.isna().sum()} already null")

        if failed.any():
            print(raw[failed].value_counts().head(10))

        data[col] = parsed


clean_grade_record_date(data)


grade_date: 0 failed to parse, 160148 already null
record_date: 0 failed to parse, 0 already null


In [95]:
def clean_grade_record_date(data):
    for col in ["grade_date", "record_date"]:
        raw = data[col]
        parsed = pd.to_datetime(raw, errors="coerce")

        failed = parsed.isna() & raw.notna()
        print(f"{col}: {failed.sum()} failed to parse, {raw.isna().sum()} already null")

        if failed.any():
            print(raw[failed].value_counts().head(10))

        data[col] = parsed
clean_grade_record_date(data)

grade_date: 0 failed to parse, 160148 already null
record_date: 0 failed to parse, 0 already null


In [96]:
def clean_violation(data):
    for col in ["violation_code", "violation_description"]:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )

    code = data["violation_code"].notna()
    desc = data["violation_description"].notna()

    print("both null (no violations recorded):", (~code & ~desc).sum())
    print("code without description:", (code & ~desc).sum())
    print("description without code:", (~code & desc).sum())

clean_violation(data)

both null (no violations recorded): 6445
code without description: 0
description without code: 0


In [97]:
def clean_critical_flag(data):
    allowed = ["Critical", "Not Critical", "Not Applicable"]

    flag = data["critical_flag"].astype("string").str.strip()
    data["critical_flag"] = flag.where(flag.isin(allowed), pd.NA).astype("category")

    data["is_critical"] = pd.Series(pd.NA, index=data.index, dtype="boolean")
    data.loc[data["critical_flag"] == "Critical", "is_critical"] = True
    data.loc[data["critical_flag"] == "Not Critical", "is_critical"] = False

clean_critical_flag(data)

In [99]:
df_check = data   # <-- your cleaned dataframe name

checks = {
    "camis never null":        df_check["camis"].notna().all(),
    "camis all digits":        df_check["camis"].astype(str).str.fullmatch(r"\d+").all(),
    "zipcode 5 digits":        df_check["zipcode"].dropna().astype(str).str.fullmatch(r"\d{5}").all(),
    "phone formatted":         df_check["phone"].dropna().astype(str).str.fullmatch(r"\d{3}-\d{3}-\d{4}").all(),
    "grade in ABCNPZ":         df_check["grade"].dropna().isin(list("ABCNPZ")).all(),
    "boro valid":              df_check["boro"].dropna().isin(
                                   ["Manhattan","Bronx","Brooklyn","Queens","Staten Island"]).all(),
    "score non-negative":      (df_check["score"].dropna() >= 0).all(),
    "critical_flag valid":     df_check["critical_flag"].dropna().isin(
                                   ["Critical","Not Critical","Not Applicable"]).all(),
    "no exact duplicates":     df_check.duplicated().sum() == 0,
    "no year-1900 dates":      (df_check["inspection_date"].dt.year == 1900).sum() == 0,
    "no future dates":         (df_check["inspection_date"] > pd.Timestamp.today()).sum() == 0,
    "lat in NYC range":        df_check["latitude"].dropna().between(40.4, 41.0).all(),
    "lon in NYC range":        df_check["longitude"].dropna().between(-74.3, -73.6).all(),
    "no empty strings":        not (df_check.select_dtypes("object")
                                     .apply(lambda s: s.str.strip() == "").any().any()),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

PASS  camis never null
PASS  camis all digits
PASS  zipcode 5 digits
PASS  phone formatted
PASS  grade in ABCNPZ
PASS  boro valid
PASS  score non-negative
PASS  critical_flag valid
PASS  no exact duplicates
PASS  no year-1900 dates
PASS  no future dates
PASS  lat in NYC range
PASS  lon in NYC range
PASS  no empty strings


In [100]:
print(len(raw), len(clean), len(rejects))
print(rejects["reject_reason"].value_counts())

NameError: name 'raw' is not defined

In [101]:
(data.isna().mean() * 100).round(1).sort_values(ascending=False)

reject_reason                  100.0
grade_date                      54.4
grade                           50.9
score                            5.8
is_critical                      2.8
violation_code                   2.2
violation_description            2.2
bin                              2.0
longitude                        1.6
census_tract                     1.6
nta                              1.6
:@computed_region_f5dn_yrer      1.6
council_district                 1.6
:@computed_region_yeji_bk3q      1.6
:@computed_region_sbqj_enih      1.6
:@computed_region_92fq_4b7q      1.6
community_board                  1.6
latitude                         1.6
inspection_program               1.3
action                           1.3
inspection_date                  1.3
cuisine_description              1.3
inspection_type                  1.3
zipcode                          1.0
bbl                              0.6
building                         0.3
phone                            0.1
b